# Lección 14: Updaters Básicos

Los **updaters** son funciones que se ejecutan en cada frame de la animación, permitiendo crear comportamientos dinámicos y reactivos. Son fundamentales para animaciones complejas donde los objetos deben responder a cambios en otros objetos o en el tiempo.

## Contenido:
1. Concepto de Updaters
2. `add_updater()` - Agregar comportamiento dinámico
3. `remove_updater()` - Eliminar updaters
4. `clear_updaters()` - Limpiar todos los updaters
5. `always_redraw()` - Reconstruir objetos automáticamente
6. `ValueTracker` con Updaters
7. `become()` - Transformaciones dinámicas
8. Updaters con dt (delta time)
9. Dependencias entre objetos
10. Casos de uso prácticos
11. Ejercicios Prácticos
12. Referencia Rápida

In [ ]:
from manim import *
import numpy as np

## 1. Concepto de Updaters

Un **updater** es una función que se ejecuta automáticamente en cada frame de renderizado. Esto permite:

- **Sincronización automática**: Un objeto sigue a otro
- **Animaciones continuas**: Movimiento basado en tiempo
- **Cálculos dinámicos**: Valores que cambian según el estado de otros objetos

### Sintaxis básica:
```python
def mi_updater(mobject):
    # Modificar mobject basado en alguna condición
    mobject.move_to(otro_objeto.get_center())

objeto.add_updater(mi_updater)
```

> ⚠️ **Importante**: Los updaters se ejecutan ~60 veces por segundo (según el frame rate)

In [ ]:
%%manim -qm -v WARNING UpdaterBasico

class UpdaterBasico(Scene):
    def construct(self):
        titulo = Text("Updater Básico", font_size=36)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # Objeto que se mueve
        circulo = Circle(radius=0.5, color=BLUE, fill_opacity=0.8)
        circulo.shift(LEFT * 3)
        
        # Objeto que sigue al círculo
        etiqueta = Text("¡Sígueme!", font_size=24, color=YELLOW)
        
        # Definir el updater: la etiqueta siempre está arriba del círculo
        def seguir_circulo(mob):
            mob.next_to(circulo, UP, buff=0.3)
        
        # Agregar el updater
        etiqueta.add_updater(seguir_circulo)
        
        self.play(Create(circulo), FadeIn(etiqueta))
        
        # Mover el círculo - la etiqueta sigue automáticamente
        self.play(circulo.animate.shift(RIGHT * 6), run_time=3)
        self.play(circulo.animate.shift(DOWN * 2), run_time=2)
        self.play(circulo.animate.move_to(ORIGIN), run_time=2)
        
        # Eliminar el updater
        etiqueta.remove_updater(seguir_circulo)
        
        # Ahora el círculo se mueve solo
        self.play(circulo.animate.shift(LEFT * 2))
        
        self.wait()

## 2. add_updater() - Agregar Comportamiento Dinámico

El método `add_updater()` recibe una función que modifica el objeto en cada frame.

### Parámetros:
```python
mobject.add_updater(
    update_function,     # Función que recibe el mobject
    index=None,          # Posición en la lista de updaters
    call_updater=True    # Si ejecutar inmediatamente
)
```

### Ejemplos de uso común:
- **Posicionamiento relativo**: Un objeto sigue a otro
- **Cambios de color**: Color que depende de posición
- **Escalado dinámico**: Tamaño que cambia según condición

In [ ]:
%%manim -qm -v WARNING MultipleUpdaters

class MultipleUpdaters(Scene):
    def construct(self):
        titulo = Text("Múltiples Updaters", font_size=36)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # Objeto principal
        punto = Dot(color=RED, radius=0.15)
        punto.move_to(LEFT * 3)
        
        # Línea que conecta al origen
        linea = always_redraw(
            lambda: Line(ORIGIN, punto.get_center(), color=YELLOW)
        )
        
        # Círculo que indica la distancia
        def crear_circulo_distancia():
            radio = np.linalg.norm(punto.get_center())
            return Circle(radius=radio, color=BLUE, stroke_opacity=0.5)
        
        circulo = always_redraw(crear_circulo_distancia)
        
        # Texto que muestra la distancia
        def crear_texto_distancia():
            distancia = np.linalg.norm(punto.get_center())
            return Text(f"d = {distancia:.2f}", font_size=24).to_edge(DOWN)
        
        texto_dist = always_redraw(crear_texto_distancia)
        
        self.play(
            Create(punto),
            Create(linea),
            Create(circulo),
            Write(texto_dist)
        )
        
        # Mover el punto
        self.play(punto.animate.move_to(RIGHT * 2 + UP * 2), run_time=2)
        self.play(punto.animate.move_to(RIGHT * 3), run_time=2)
        self.play(punto.animate.move_to(DOWN * 2), run_time=2)
        self.play(punto.animate.move_to(LEFT * 2 + UP), run_time=2)
        
        self.wait()

## 3. remove_updater() y clear_updaters()

### Eliminar updaters específicos:
```python
mobject.remove_updater(update_function)  # Elimina un updater específico
```

### Eliminar todos los updaters:
```python
mobject.clear_updaters()  # Elimina todos los updaters del objeto
```

Útil cuando necesitas que un objeto deje de seguir a otro o cambiar su comportamiento.

In [ ]:
%%manim -qm -v WARNING ControlUpdaters

class ControlUpdaters(Scene):
    def construct(self):
        titulo = Text("Control de Updaters", font_size=36)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # Crear objetos
        cuadrado = Square(side_length=1, color=BLUE, fill_opacity=0.7)
        cuadrado.shift(LEFT * 3)
        
        triangulo = Triangle(color=RED, fill_opacity=0.7)
        triangulo.scale(0.5)
        
        circulo = Circle(radius=0.3, color=GREEN, fill_opacity=0.7)
        
        # Updaters para seguir al cuadrado
        def seguir_derecha(mob):
            mob.next_to(cuadrado, RIGHT, buff=0.5)
        
        def seguir_abajo(mob):
            mob.next_to(cuadrado, DOWN, buff=0.5)
        
        triangulo.add_updater(seguir_derecha)
        circulo.add_updater(seguir_abajo)
        
        self.play(Create(cuadrado), Create(triangulo), Create(circulo))
        
        # Fase 1: Ambos siguen
        texto1 = Text("Ambos siguen al cuadrado", font_size=20).to_edge(DOWN)
        self.play(Write(texto1))
        self.play(cuadrado.animate.shift(RIGHT * 3), run_time=2)
        
        # Fase 2: Eliminar updater del triángulo
        self.play(FadeOut(texto1))
        texto2 = Text("Triángulo desvinculado", font_size=20).to_edge(DOWN)
        self.play(Write(texto2))
        
        triangulo.remove_updater(seguir_derecha)
        self.play(cuadrado.animate.shift(UP * 2), run_time=2)
        
        # Fase 3: Limpiar todos los updaters
        self.play(FadeOut(texto2))
        texto3 = Text("Todos los updaters eliminados", font_size=20).to_edge(DOWN)
        self.play(Write(texto3))
        
        circulo.clear_updaters()
        self.play(cuadrado.animate.shift(LEFT * 3), run_time=2)
        
        self.wait()

## 4. always_redraw() - Reconstruir Objetos Automáticamente

`always_redraw()` es una función muy potente que reconstruye completamente un Mobject en cada frame.

### Sintaxis:
```python
objeto = always_redraw(lambda: crear_objeto())
```

### Ventajas:
- Perfecto para objetos que dependen de otros (líneas, flechas, texto dinámico)
- Más limpio que usar `add_updater()` para ciertos casos
- El objeto se recrea completamente, evitando inconsistencias

### Uso común:
- Líneas entre puntos móviles
- Texto que muestra valores cambiantes
- Formas que dependen de parámetros variables

In [ ]:
%%manim -qm -v WARNING AlwaysRedrawDemo

class AlwaysRedrawDemo(Scene):
    def construct(self):
        titulo = Text("always_redraw() en Acción", font_size=36)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # Dos puntos que se pueden mover
        punto_a = Dot(color=RED, radius=0.15).move_to(LEFT * 3 + UP)
        punto_b = Dot(color=BLUE, radius=0.15).move_to(RIGHT * 2 + DOWN)
        
        etiqueta_a = Text("A", font_size=20, color=RED).next_to(punto_a, UP)
        etiqueta_b = Text("B", font_size=20, color=BLUE).next_to(punto_b, UP)
        
        # Línea que siempre conecta A con B
        linea = always_redraw(
            lambda: Line(
                punto_a.get_center(), 
                punto_b.get_center(), 
                color=YELLOW,
                stroke_width=3
            )
        )
        
        # Punto medio que siempre está en el centro de la línea
        punto_medio = always_redraw(
            lambda: Dot(
                (punto_a.get_center() + punto_b.get_center()) / 2,
                color=GREEN,
                radius=0.1
            )
        )
        
        # Texto con la distancia
        texto_distancia = always_redraw(
            lambda: MathTex(
                f"d = {np.linalg.norm(punto_b.get_center() - punto_a.get_center()):.2f}",
                font_size=28
            ).to_edge(DOWN)
        )
        
        self.play(
            Create(punto_a), Create(punto_b),
            FadeIn(etiqueta_a), FadeIn(etiqueta_b),
            Create(linea), Create(punto_medio),
            Write(texto_distancia)
        )
        
        # Actualizar etiquetas para que sigan a los puntos
        etiqueta_a.add_updater(lambda m: m.next_to(punto_a, UP))
        etiqueta_b.add_updater(lambda m: m.next_to(punto_b, UP))
        
        # Mover los puntos
        self.play(punto_a.animate.move_to(LEFT * 2 + DOWN * 2), run_time=2)
        self.play(punto_b.animate.move_to(RIGHT * 3 + UP * 2), run_time=2)
        self.play(
            punto_a.animate.move_to(UP * 2),
            punto_b.animate.move_to(DOWN * 2),
            run_time=2
        )
        
        self.wait()

## 5. ValueTracker con Updaters

`ValueTracker` es un Mobject especial que almacena un valor numérico y puede ser animado. Es fundamental para crear animaciones paramétricas.

### Métodos principales:
```python
valor = ValueTracker(0)           # Crear con valor inicial
valor.get_value()                 # Obtener valor actual
valor.set_value(5)                # Establecer valor
valor.animate.set_value(10)       # Animar cambio de valor
valor.increment_value(1)          # Incrementar valor
```

### Patrón típico:
```python
t = ValueTracker(0)
obj.add_updater(lambda m: m.move_to(RIGHT * t.get_value()))
self.play(t.animate.set_value(3))
```

In [ ]:
%%manim -qm -v WARNING ValueTrackerConUpdater

class ValueTrackerConUpdater(Scene):
    def construct(self):
        titulo = Text("ValueTracker + Updaters", font_size=36)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # ValueTracker para controlar el ángulo
        angulo = ValueTracker(0)
        
        # Configuración
        radio = 2
        centro = ORIGIN
        
        # Círculo de referencia
        circulo_ref = Circle(radius=radio, color=GRAY, stroke_opacity=0.5)
        
        # Punto que se mueve en el círculo
        punto = always_redraw(
            lambda: Dot(
                centro + radio * np.array([
                    np.cos(angulo.get_value()),
                    np.sin(angulo.get_value()),
                    0
                ]),
                color=RED,
                radius=0.15
            )
        )
        
        # Línea desde el centro al punto
        linea = always_redraw(
            lambda: Line(
                centro,
                centro + radio * np.array([
                    np.cos(angulo.get_value()),
                    np.sin(angulo.get_value()),
                    0
                ]),
                color=YELLOW
            )
        )
        
        # Arco que muestra el ángulo
        arco = always_redraw(
            lambda: Arc(
                radius=0.5,
                start_angle=0,
                angle=angulo.get_value(),
                color=GREEN
            ) if angulo.get_value() > 0.01 else VGroup()
        )
        
        # Texto con el ángulo en grados
        texto_angulo = always_redraw(
            lambda: MathTex(
                f"\\theta = {np.degrees(angulo.get_value()):.1f}°",
                font_size=32
            ).to_edge(DOWN)
        )
        
        self.play(Create(circulo_ref))
        self.add(linea, arco, punto, texto_angulo)
        
        # Animar el ángulo
        self.play(angulo.animate.set_value(PI/2), run_time=2)
        self.play(angulo.animate.set_value(PI), run_time=2)
        self.play(angulo.animate.set_value(2*PI), run_time=3)
        
        self.wait()

## 6. become() - Transformaciones Dinámicas

El método `become()` permite que un Mobject adopte la forma de otro Mobject, manteniendo su identidad.

### Sintaxis:
```python
mobject.become(otro_mobject)
```

### Diferencia con Transform:
- `Transform(a, b)`: Animación gradual de `a` hacia `b`
- `a.become(b)`: Cambio instantáneo, útil en updaters

### Uso en updaters:
```python
def mi_updater(mob):
    nuevo = crear_forma_segun_condicion()
    mob.become(nuevo)
```

In [ ]:
%%manim -qm -v WARNING BecomeConUpdater

class BecomeConUpdater(Scene):
    def construct(self):
        titulo = Text("become() con Updaters", font_size=36)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # ValueTracker para controlar el número de lados
        n_lados = ValueTracker(3)
        
        # Polígono que cambia según el número de lados
        poligono = RegularPolygon(n=3, color=BLUE, fill_opacity=0.7)
        poligono.scale(1.5)
        
        def actualizar_poligono(mob):
            n = int(n_lados.get_value())
            nuevo = RegularPolygon(n=n, color=BLUE, fill_opacity=0.7)
            nuevo.scale(1.5)
            mob.become(nuevo)
        
        poligono.add_updater(actualizar_poligono)
        
        # Texto que muestra el número de lados
        texto = always_redraw(
            lambda: Text(
                f"Lados: {int(n_lados.get_value())}",
                font_size=28
            ).to_edge(DOWN)
        )
        
        self.play(Create(poligono), Write(texto))
        
        # Animar cambio de lados
        for n in [4, 5, 6, 8, 10, 12]:
            self.play(n_lados.animate.set_value(n), run_time=1)
            self.wait(0.3)
        
        # Volver a triángulo
        self.play(n_lados.animate.set_value(3), run_time=1.5)
        
        poligono.clear_updaters()
        self.wait()

## 7. Updaters con dt (Delta Time)

Los updaters pueden recibir un segundo parámetro `dt` que representa el tiempo transcurrido desde el último frame.

### Sintaxis:
```python
def updater_con_dt(mobject, dt):
    # dt ≈ 1/frame_rate segundos (típicamente ~0.016s a 60fps)
    mobject.shift(RIGHT * velocidad * dt)

objeto.add_updater(updater_con_dt)
```

### Aplicaciones:
- **Velocidad constante**: Movimiento independiente del frame rate
- **Física simple**: Gravedad, fricción
- **Acumuladores**: Contadores de tiempo
- **Oscilaciones**: Movimiento sinusoidal basado en tiempo

In [ ]:
%%manim -qm -v WARNING UpdaterConDeltaTime

class UpdaterConDeltaTime(Scene):
    def construct(self):
        titulo = Text("Updaters con dt (Delta Time)", font_size=36)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # Partícula con velocidad constante
        particula = Dot(color=RED, radius=0.2)
        particula.move_to(LEFT * 5)
        
        velocidad = 2  # unidades por segundo
        
        def mover_derecha(mob, dt):
            mob.shift(RIGHT * velocidad * dt)
        
        particula.add_updater(mover_derecha)
        
        texto_vel = Text(f"Velocidad: {velocidad} u/s", font_size=24).to_edge(DOWN)
        self.add(particula, texto_vel)
        
        self.wait(5)  # La partícula se mueve 10 unidades en 5 segundos
        
        particula.clear_updaters()
        self.wait(0.5)
        
        # Demostración con oscilación
        self.play(FadeOut(particula), FadeOut(texto_vel))
        
        tiempo_total = ValueTracker(0)
        
        # Partícula oscilante
        oscilador = Dot(color=BLUE, radius=0.2)
        
        def oscilar(mob, dt):
            tiempo_total.increment_value(dt)
            t = tiempo_total.get_value()
            mob.move_to(np.array([
                3 * np.cos(2 * t),
                2 * np.sin(3 * t),
                0
            ]))
        
        oscilador.add_updater(oscilar)
        
        # Rastro del oscilador
        rastro = TracedPath(oscilador.get_center, stroke_color=YELLOW, stroke_width=2)
        
        texto_osc = Text("Curva de Lissajous", font_size=24).to_edge(DOWN)
        self.add(oscilador, rastro, texto_osc)
        
        self.wait(6)
        
        oscilador.clear_updaters()
        self.wait()

## 8. Dependencias entre Objetos

Los updaters permiten crear cadenas de dependencias donde múltiples objetos reaccionan a cambios en otros.

### Patrones comunes:

**1. Seguimiento simple:**
```python
etiqueta.add_updater(lambda m: m.next_to(objeto, UP))
```

**2. Dependencia calculada:**
```python
def actualizar_barra(mob):
    mob.set_width(valor.get_value() * escala)
```

**3. Dependencia múltiple:**
```python
linea = always_redraw(lambda: Line(punto_a.get_center(), punto_b.get_center()))
```

In [ ]:
%%manim -qm -v WARNING DependenciasObjetos

class DependenciasObjetos(Scene):
    def construct(self):
        titulo = Text("Dependencias entre Objetos", font_size=36)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # Punto principal (controlador)
        punto_control = Dot(color=RED, radius=0.2)
        punto_control.move_to(LEFT * 2)
        
        # Objetos dependientes
        # 1. Etiqueta que sigue al punto
        etiqueta = Text("P", font_size=24, color=RED)
        etiqueta.add_updater(lambda m: m.next_to(punto_control, UR, buff=0.1))
        
        # 2. Círculo cuyo radio depende de la distancia al origen
        circulo = always_redraw(
            lambda: Circle(
                radius=np.linalg.norm(punto_control.get_center()),
                color=BLUE,
                stroke_opacity=0.5
            )
        )
        
        # 3. Flecha desde el origen al punto
        flecha = always_redraw(
            lambda: Arrow(
                ORIGIN, 
                punto_control.get_center(), 
                color=YELLOW,
                buff=0
            )
        )
        
        # 4. Texto con coordenadas
        coords = always_redraw(
            lambda: MathTex(
                f"({punto_control.get_center()[0]:.1f}, {punto_control.get_center()[1]:.1f})",
                font_size=28
            ).to_edge(DOWN)
        )
        
        # 5. Proyecciones en los ejes
        proy_x = always_redraw(
            lambda: DashedLine(
                punto_control.get_center(),
                np.array([punto_control.get_center()[0], 0, 0]),
                color=GREEN
            )
        )
        
        proy_y = always_redraw(
            lambda: DashedLine(
                punto_control.get_center(),
                np.array([0, punto_control.get_center()[1], 0]),
                color=GREEN
            )
        )
        
        # Ejes de referencia
        ejes = Axes(
            x_range=[-4, 4, 1],
            y_range=[-3, 3, 1],
            x_length=8,
            y_length=6,
            tips=False
        ).set_opacity(0.3)
        
        self.add(ejes)
        self.play(
            Create(punto_control),
            FadeIn(etiqueta),
            Create(circulo),
            Create(flecha),
            Write(coords),
            Create(proy_x),
            Create(proy_y)
        )
        
        # Mover el punto - todo se actualiza automáticamente
        self.play(punto_control.animate.move_to(RIGHT * 2 + UP * 2), run_time=2)
        self.play(punto_control.animate.move_to(RIGHT * 3), run_time=1.5)
        self.play(punto_control.animate.move_to(DOWN * 2 + LEFT), run_time=2)
        self.play(punto_control.animate.move_to(UP + LEFT * 2), run_time=2)
        
        self.wait()

## 9. Casos de Uso Prácticos

### Caso 1: Gráfica de función con tangente dinámica
Una línea tangente que sigue un punto móvil sobre una curva.

In [ ]:
%%manim -qm -v WARNING TangenteDinamica

class TangenteDinamica(Scene):
    def construct(self):
        titulo = Text("Tangente Dinámica", font_size=36)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # Sistema de ejes
        axes = Axes(
            x_range=[-1, 5, 1],
            y_range=[-1, 4, 1],
            x_length=7,
            y_length=5,
            tips=True
        )
        axes.shift(DOWN * 0.5 + LEFT)
        
        # Función: f(x) = 0.3x² - 0.5x + 1
        def f(x):
            return 0.3 * x**2 - 0.5 * x + 1
        
        # Derivada: f'(x) = 0.6x - 0.5
        def f_prima(x):
            return 0.6 * x - 0.5
        
        curva = axes.plot(f, x_range=[0, 4.5], color=BLUE)
        
        # ValueTracker para la posición x
        x_val = ValueTracker(0.5)
        
        # Punto sobre la curva
        punto = always_redraw(
            lambda: Dot(
                axes.c2p(x_val.get_value(), f(x_val.get_value())),
                color=RED
            )
        )
        
        # Línea tangente
        def get_tangente():
            x = x_val.get_value()
            y = f(x)
            pendiente = f_prima(x)
            
            # Puntos de la tangente
            x1, x2 = x - 1, x + 1
            y1 = y + pendiente * (x1 - x)
            y2 = y + pendiente * (x2 - x)
            
            return Line(
                axes.c2p(x1, y1),
                axes.c2p(x2, y2),
                color=YELLOW
            )
        
        tangente = always_redraw(get_tangente)
        
        # Texto con la pendiente
        texto_pendiente = always_redraw(
            lambda: MathTex(
                f"f'({x_val.get_value():.1f}) = {f_prima(x_val.get_value()):.2f}",
                font_size=28
            ).to_edge(DOWN)
        )
        
        self.play(Create(axes), Create(curva))
        self.play(Create(punto), Create(tangente), Write(texto_pendiente))
        
        # Animar el punto a lo largo de la curva
        self.play(x_val.animate.set_value(4), run_time=5, rate_func=smooth)
        self.play(x_val.animate.set_value(1), run_time=3, rate_func=smooth)
        
        self.wait()

### Caso 2: Barra de progreso animada
Una barra que se llena gradualmente con texto de porcentaje sincronizado.

In [ ]:
%%manim -qm -v WARNING BarraProgreso

class BarraProgreso(Scene):
    def construct(self):
        titulo = Text("Barra de Progreso", font_size=36)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # ValueTracker para el progreso (0 a 1)
        progreso = ValueTracker(0)
        
        ancho_total = 6
        alto = 0.5
        
        # Fondo de la barra
        fondo = Rectangle(
            width=ancho_total,
            height=alto,
            color=GRAY,
            fill_opacity=0.3,
            stroke_width=2
        )
        
        # Barra de progreso (se llena)
        def get_barra_relleno():
            ancho_actual = max(0.01, ancho_total * progreso.get_value())
            barra = Rectangle(
                width=ancho_actual,
                height=alto,
                color=GREEN,
                fill_opacity=0.8,
                stroke_width=0
            )
            barra.align_to(fondo, LEFT)
            return barra
        
        relleno = always_redraw(get_barra_relleno)
        
        # Texto de porcentaje
        texto_porcentaje = always_redraw(
            lambda: Text(
                f"{int(progreso.get_value() * 100)}%",
                font_size=32,
                color=WHITE
            ).next_to(fondo, DOWN, buff=0.5)
        )
        
        # Etiqueta descriptiva
        etiqueta = Text("Cargando...", font_size=24)
        etiqueta.next_to(fondo, UP, buff=0.5)
        
        self.play(Create(fondo), FadeIn(etiqueta))
        self.add(relleno, texto_porcentaje)
        
        # Animar el progreso
        self.play(
            progreso.animate.set_value(1),
            run_time=4,
            rate_func=smooth
        )
        
        # Cambiar etiqueta a "Completado"
        nueva_etiqueta = Text("¡Completado!", font_size=24, color=GREEN)
        nueva_etiqueta.next_to(fondo, UP, buff=0.5)
        self.play(Transform(etiqueta, nueva_etiqueta))
        
        self.wait()

### Caso 3: Reloj analógico animado
Un reloj con manecillas que se mueven usando updaters basados en tiempo.

In [ ]:
%%manim -qm -v WARNING RelojAnimado

class RelojAnimado(Scene):
    def construct(self):
        titulo = Text("Reloj Animado", font_size=36)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # Cara del reloj
        cara = Circle(radius=2, color=WHITE, stroke_width=4)
        centro = Dot(color=WHITE, radius=0.08)
        
        # Marcas de las horas
        marcas = VGroup()
        for i in range(12):
            angulo = i * PI / 6 - PI / 2
            inicio = 1.7 * np.array([np.cos(angulo), np.sin(angulo), 0])
            fin = 2 * np.array([np.cos(angulo), np.sin(angulo), 0])
            linea = Line(inicio, fin, color=WHITE, stroke_width=3)
            marcas.add(linea)
        
        # Tiempo acumulado
        tiempo = ValueTracker(0)
        
        # Velocidad: 1 segundo de animación = 1 minuto del reloj
        # La manecilla de segundos da una vuelta completa cada segundo de animación
        
        # Manecilla de segundos
        def get_segundos():
            t = tiempo.get_value()
            angulo = -t * 2 * PI + PI / 2  # Sentido horario
            return Line(
                ORIGIN,
                1.5 * np.array([np.cos(angulo), np.sin(angulo), 0]),
                color=RED,
                stroke_width=2
            )
        
        # Manecilla de minutos (60 veces más lenta)
        def get_minutos():
            t = tiempo.get_value()
            angulo = -t * 2 * PI / 60 + PI / 2
            return Line(
                ORIGIN,
                1.3 * np.array([np.cos(angulo), np.sin(angulo), 0]),
                color=BLUE,
                stroke_width=4
            )
        
        # Manecilla de horas (720 veces más lenta)
        def get_horas():
            t = tiempo.get_value()
            angulo = -t * 2 * PI / 720 + PI / 2
            return Line(
                ORIGIN,
                0.9 * np.array([np.cos(angulo), np.sin(angulo), 0]),
                color=GREEN,
                stroke_width=6
            )
        
        segundos = always_redraw(get_segundos)
        minutos = always_redraw(get_minutos)
        horas = always_redraw(get_horas)
        
        self.play(Create(cara), Create(marcas), Create(centro))
        self.add(horas, minutos, segundos)
        
        # Animar 5 "minutos" del reloj
        self.play(tiempo.animate.set_value(5), run_time=5, rate_func=linear)
        
        self.wait()

## 10. Ejercicios Prácticos

### Ejercicio 1: Sistema Solar Simplificado
Crea un sol en el centro con planetas orbitando usando updaters con `dt`. Cada planeta debe tener velocidad angular diferente.

### Ejercicio 2: Indicador de Velocímetro
Crea un velocímetro semicircular con una aguja que responde a un `ValueTracker`. Incluye marcas de velocidad y texto dinámico.

### Ejercicio 3: Péndulo Simple
Simula un péndulo con una cuerda y una masa que oscila usando la fórmula:
$$\theta(t) = \theta_0 \cos(\sqrt{g/L} \cdot t)$$

In [ ]:
%%manim -qm -v WARNING Ejercicio1_SistemaSolar

class Ejercicio1_SistemaSolar(Scene):
    def construct(self):
        titulo = Text("Sistema Solar Simplificado", font_size=32)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # Sol
        sol = Dot(color=YELLOW, radius=0.3)
        sol.set_fill(YELLOW, opacity=1)
        
        # Tiempo
        tiempo = ValueTracker(0)
        
        # Datos de planetas: (radio_orbita, color, radio_planeta, velocidad_angular)
        planetas_data = [
            (1.0, GRAY, 0.08, 4.0),    # Mercurio
            (1.5, ORANGE, 0.10, 3.0),  # Venus
            (2.0, BLUE, 0.12, 2.0),    # Tierra
            (2.7, RED, 0.10, 1.5),     # Marte
        ]
        
        orbitas = VGroup()
        planetas = VGroup()
        
        for radio, color, tam, vel in planetas_data:
            # Órbita (círculo de referencia)
            orbita = Circle(radius=radio, color=WHITE, stroke_opacity=0.2)
            orbitas.add(orbita)
            
            # Planeta
            planeta = always_redraw(
                lambda r=radio, c=color, t=tam, v=vel: Dot(
                    r * np.array([
                        np.cos(tiempo.get_value() * v),
                        np.sin(tiempo.get_value() * v),
                        0
                    ]),
                    color=c,
                    radius=t
                )
            )
            planetas.add(planeta)
        
        self.play(Create(sol), Create(orbitas))
        self.add(*planetas)
        
        # Animar el sistema
        self.play(tiempo.animate.set_value(2 * PI), run_time=6, rate_func=linear)
        
        self.wait()

In [ ]:
%%manim -qm -v WARNING Ejercicio2_Velocimetro

class Ejercicio2_Velocimetro(Scene):
    def construct(self):
        titulo = Text("Velocímetro", font_size=32)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # Arco del velocímetro
        arco = Arc(
            radius=2,
            start_angle=PI,
            angle=-PI,
            color=WHITE,
            stroke_width=8
        )
        arco.shift(DOWN * 0.5)
        
        # Marcas de velocidad (0-200 km/h)
        marcas = VGroup()
        etiquetas = VGroup()
        
        for i in range(11):
            angulo = PI - (i * PI / 10)
            # Marca
            inicio = 1.7 * np.array([np.cos(angulo), np.sin(angulo), 0]) + DOWN * 0.5
            fin = 2 * np.array([np.cos(angulo), np.sin(angulo), 0]) + DOWN * 0.5
            marca = Line(inicio, fin, color=WHITE, stroke_width=2)
            marcas.add(marca)
            
            # Etiqueta
            pos = 2.3 * np.array([np.cos(angulo), np.sin(angulo), 0]) + DOWN * 0.5
            etiqueta = Text(str(i * 20), font_size=18)
            etiqueta.move_to(pos)
            etiquetas.add(etiqueta)
        
        # Velocidad (0 a 1, donde 1 = 200 km/h)
        velocidad = ValueTracker(0)
        
        # Aguja
        def get_aguja():
            v = velocidad.get_value()
            angulo = PI - v * PI
            return Line(
                DOWN * 0.5,
                1.5 * np.array([np.cos(angulo), np.sin(angulo), 0]) + DOWN * 0.5,
                color=RED,
                stroke_width=6
            )
        
        aguja = always_redraw(get_aguja)
        centro = Dot(DOWN * 0.5, color=RED, radius=0.1)
        
        # Texto de velocidad
        texto_vel = always_redraw(
            lambda: Text(
                f"{int(velocidad.get_value() * 200)} km/h",
                font_size=36
            ).move_to(DOWN * 2.5)
        )
        
        self.play(Create(arco), Create(marcas), Write(etiquetas))
        self.add(aguja, centro, texto_vel)
        
        # Animar aceleración y frenado
        self.play(velocidad.animate.set_value(0.6), run_time=2, rate_func=smooth)
        self.play(velocidad.animate.set_value(0.9), run_time=1.5, rate_func=smooth)
        self.play(velocidad.animate.set_value(0.3), run_time=2, rate_func=smooth)
        self.play(velocidad.animate.set_value(0), run_time=1.5, rate_func=smooth)
        
        self.wait()

In [ ]:
%%manim -qm -v WARNING Ejercicio3_Pendulo

class Ejercicio3_Pendulo(Scene):
    def construct(self):
        titulo = Text("Péndulo Simple", font_size=32)
        titulo.to_edge(UP)
        self.play(Write(titulo))
        
        # Parámetros del péndulo
        L = 2.5  # Longitud
        g = 9.8  # Gravedad
        theta_0 = PI / 6  # Ángulo inicial (30°)
        omega = np.sqrt(g / L)  # Frecuencia angular
        
        # Punto de suspensión
        pivote = Dot(UP * 2, color=WHITE, radius=0.1)
        soporte = Line(UP * 2 + LEFT * 0.5, UP * 2 + RIGHT * 0.5, color=WHITE, stroke_width=4)
        
        # Tiempo
        tiempo = ValueTracker(0)
        
        # Función del ángulo
        def get_theta():
            t = tiempo.get_value()
            return theta_0 * np.cos(omega * t)
        
        # Cuerda
        def get_cuerda():
            theta = get_theta()
            fin = UP * 2 + L * np.array([np.sin(theta), -np.cos(theta), 0])
            return Line(UP * 2, fin, color=YELLOW, stroke_width=3)
        
        cuerda = always_redraw(get_cuerda)
        
        # Masa
        def get_masa():
            theta = get_theta()
            pos = UP * 2 + L * np.array([np.sin(theta), -np.cos(theta), 0])
            return Dot(pos, color=BLUE, radius=0.2)
        
        masa = always_redraw(get_masa)
        
        # Rastro de la masa
        def get_pos_masa():
            theta = get_theta()
            return UP * 2 + L * np.array([np.sin(theta), -np.cos(theta), 0])
        
        rastro = TracedPath(get_pos_masa, stroke_color=RED, stroke_opacity=0.5, stroke_width=2)
        
        # Texto con el ángulo
        texto_angulo = always_redraw(
            lambda: MathTex(
                f"\\theta = {np.degrees(get_theta()):.1f}°",
                font_size=28
            ).to_edge(DOWN)
        )
        
        self.play(Create(soporte), Create(pivote))
        self.add(cuerda, masa, rastro, texto_angulo)
        
        # Oscilar por varios períodos
        T = 2 * PI / omega  # Período
        self.play(tiempo.animate.set_value(3 * T), run_time=6, rate_func=linear)
        
        self.wait()

## 11. Referencia Rápida

### Métodos de Updaters

| Método | Descripción | Ejemplo |
|--------|-------------|---------|
| `add_updater(func)` | Agrega función que se ejecuta cada frame | `obj.add_updater(lambda m: m.next_to(otro, UP))` |
| `remove_updater(func)` | Elimina un updater específico | `obj.remove_updater(mi_updater)` |
| `clear_updaters()` | Elimina todos los updaters | `obj.clear_updaters()` |
| `suspend_updating()` | Pausa temporalmente los updaters | `obj.suspend_updating()` |
| `resume_updating()` | Reanuda los updaters | `obj.resume_updating()` |

### Funciones de Conveniencia

| Función | Descripción |
|---------|-------------|
| `always_redraw(func)` | Reconstruye el Mobject cada frame |
| `always(func, *args)` | Ejecuta función cada frame (sin reconstruir) |
| `f_always(obj, method, *args)` | Llama método del objeto cada frame |

### ValueTracker

```python
t = ValueTracker(0)             # Crear con valor inicial
t.get_value()                   # Obtener valor
t.set_value(5)                  # Establecer valor
t.increment_value(0.1)          # Incrementar
t.animate.set_value(10)         # Animación
```

### Patrones Comunes

**Seguimiento de posición:**
```python
etiqueta.add_updater(lambda m: m.next_to(objeto, UP))
```

**Línea entre puntos:**
```python
linea = always_redraw(lambda: Line(p1.get_center(), p2.get_center()))
```

**Texto dinámico:**
```python
texto = always_redraw(lambda: Text(f"x = {valor.get_value():.2f}"))
```

**Movimiento con velocidad:**
```python
obj.add_updater(lambda m, dt: m.shift(RIGHT * velocidad * dt))
```

**Transformación con become:**
```python
def actualizar(mob):
    nuevo = crear_forma(parametro)
    mob.become(nuevo)
```

### Consejos Importantes

1. **Limpiar updaters**: Siempre llama `clear_updaters()` cuando ya no necesites el comportamiento
2. **Evitar referencias circulares**: No hagas que A siga a B y B siga a A
3. **Usar lambdas con cuidado**: Captura variables con valores por defecto si usas en bucles
4. **Preferir `always_redraw`**: Para objetos geométricos que dependen de otros
5. **Usar `dt` para física**: Mantiene velocidades consistentes independiente del frame rate

---

## 📚 Recursos Adicionales

### Documentación Oficial
- [Updaters](https://docs.manim.community/en/stable/tutorials/quickstart.html#making-objects-respond-to-updates)
- [ValueTracker](https://docs.manim.community/en/stable/reference/manim.mobject.value_tracker.ValueTracker.html)
- [always_redraw](https://docs.manim.community/en/stable/reference/manim.mobject.mobject.always_redraw.html)

### Conceptos Relacionados
- Rate Functions para controlar la velocidad de animaciones
- Transform vs become para cambios de forma
- TracedPath para dejar rastros

---

**Lección 14 completada ✓**

Este es el final del curso. ¡Felicidades por completar las 15 lecciones del Curso Manim Professional!